# Lektion 12 - Reduktion af chat-historik med agent-skriveblok

Denne notesbog demonstrerer, hvordan man håndterer kontekst i lange samtaler ved hjælp af Microsoft Agent Framework. Efterhånden som samtaler vokser, øges tokens antal — til sidst overstiger det modellens kontekstvindue. Vi løser dette med et **kontekstsammendragningsmønster** og en **agent-skriveblok** til vedvarende hukommelse.

## Hvad du vil lære:
1. **Hvorfor kontekststyring er vigtigt**: Forståelse af tokengrænser og kontekstvinduer
2. **Kontekstbevidste agenter**: Opbygning af agenter, der styrer deres egen samtalekontekst
3. **Kontekstsammendragningsmønster**: Brug af værktøjer til at kondensere samtalehistorik
4. **Agent-skriveblok**: Vedvarende hukommelse, der overlever kontekstreduktion

## Forudsætninger:
- Azure OpenAI opsætning med konfigurerede miljøvariabler
- Forståelse af grundlæggende agentbegreber fra tidligere lektioner


## Opsætning


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Hvorfor kontekststyring er vigtigt

Hver LLM har et begrænset **kontekstvindue** — det maksimale antal tokens, den kan bearbejde i en enkelt forespørgsel. Efterhånden som en flertrins-samtale skrider frem:

- **Token-tællingen vokser lineært** med hver brugermeddelelse og assistentens svar.
- **Prompt-tokens dominerer omkostningerne**, fordi hele historikken sendes igen ved hvert trin.
- Til sidst **overskrider samtalen kontekstvinduet**, og modellen enten afkorter eller giver fejl.

### Strategier til håndtering af kontekst

| Strategi | Hvordan det virker | Afvejning |
|---|---|---|
| **Afkortning** | Drop ældste beskeder | Mister tidlig kontekst |
| **Opsummering** | Komprimer ældre beskeder til et sammendrag | Noget detaljer går tabt, men hovedpunkterne bevares |
| **Scratchpad / Ekstern hukommelse** | Gem vigtige fakta uden for samtalen | Kræver værktøjskald, men overlever enhver nedskæring |

I denne notesbog kombinerer vi **opsummering** med et **scratchpad-værktøj**, så agenten kan bevare kontinuitet, selv når samtalehistorikken er kondenseret.


## Oprettelse af en kontekstbevidst agent


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulering af en lang samtale

Lad os gennemgå en samtale med flere omgange for at se, hvordan kontekst akkumuleres. Agenten skal bevare nøgleoplysninger (præferencer, budget, rejsedatoer) på tværs af omgange og demonstrere sammenhæng.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Bemærk, hvordan agenten bevarer konteksten fra tidligere runder — den ved noget om Japan, sushi, templer, fotografering, budgettet på $3000, solo rejse og tidsrammen i april. I en kort samtale fungerer dette godt, men efterhånden som samtalen vokser, bliver det dyrt at sende hele historikken igen.

Lad os fortsætte samtalen med flere runder for at se, hvordan konteksten ophobes:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Contextopsummeringsmønster

Efterhånden som samtalen vokser, kan vi bruge et **opsummeringsværktøj** til at kondensere den opsamlede kontekst til et kompakt format. Agenten kalder dette værktøj for at registrere nøglepræferencer, så selvom ældre beskeder fjernes, bevares den væsentlige information.

Dette mønster er byggesten for mere sofistikeret reduktion af historik:
1. Agenten identificerer nøglefakta fra samtalen
2. Den kalder opsummeringsværktøjet for at bevare dem
3. Ældre beskeder kan trygt fjernes, fordi opsummeringen fanger det væsentlige

Nedenfor definerer vi et `summarize_preferences` værktøj, som agenten kan kalde for at registrere et kompakt resume af, hvad den har lært.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Resumé

I denne lektion lærte du, hvordan man håndterer kontekst i langvarige agent-samtaler ved hjælp af Microsoft Agent Framework:

### Centrale Begreber
- **Kontekstvinduer er begrænsede** — hver token i samtalehistorikken koster penge og tæller mod grænsen.
- **Opsummeringsværktøjer** lader agenten sammenfatte opsamlet kontekst til kompakte opsummeringer, hvilket reducerer tokenforbruget samtidig med, at væsentlig information bevares.
- **Agent-scratchpads** giver vedvarende ekstern hukommelse, som overlever enhver samtalereduktion.

### Hvad Du Har Bygget
- En **kontekstbevidst agent**, som opretholder kontinuitet på tværs af flertrins-samtaler
- Et **opsummeringsværktøj** (`summarize_preferences`), der optager nøgleoplysninger om brugeren i et kompakt format
- En **flertrins-samtale**, der demonstrerer kontekstbevarelse og håndtering af ændringer

### Anvendelser i Virkeligheden
- **Kundeservice-bots**: Husker præferencer på tværs af lange supportsessioner
- **Personlige assistenter**: Følger igangværende projekter uden at skulle forklare kontekst igen
- **Undervisningstutorer**: Opretholder elevens fremskridt over mange interaktioner

### Næste Skridt
- Implementer et fuldt scratchpad-værktøj med filbaseret vedholdenhed
- Tilføj automatisk historikforkortelse efter opsummering
- Kombiner med vektordatabaser til semantisk hukommelsessøgning
- Byg agenter, der kan genoptage samtaler dage senere med fuld kontekst


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, bedes du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
